# Important info

## Faostat crop production data
- Bulk faostat csv: https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip
- Browse/schema interactively: https://www.fao.org/faostat/en/#data/QCL (the "Definitions and standards" and "Metadata" tabs there document item codes, elements, and flags).
- Normalized schema (one row per area × item × element × year): Area Code, Area, Item Code (CPC), Item, Element Code, Element, Year, Unit, Value, Flag. The three elements you care about are Area harvested (ha), Production (t), and Yield (hg/ha) — divide yield by 10,000 to get kg/m². For a global figure, filter Area = 'World'; for a high-yield ceiling, filter a top producer (e.g. Netherlands, potatoes).

## USDA FoodData Central — energy (and full nutrient) densities
- SR Legacy foods (CSV): https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_sr_legacy_food_csv_2018-04.zip (6.7 MB zipped, ~54 MB unzipped). This is the one I'd start with, it has broad coverage of raw commodities (potato raw, sweet potato raw, peanuts, etc.) with per-100 g energy and refuse fractions.
- Foundation Foods (CSV): https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_foundation_food_csv_2026-04-30.zip (3.7 MB) — newer and more granular, but narrower coverage. Good for cross-checking a few crops, less good as the base.
- Schema/field definitions (PDF): https://fdc.nal.usda.gov/docs/Download_Field_Descriptions_Oct2020.pdf
- Schema shape: it's relational, several CSVs. The join for energy is food.csv (fdc_id, description, data_type) → food_nutrient.csv (fdc_id, nutrient_id, amount) → nutrient.csv (id, name, unit_name), filtering nutrient.name = 'Energy' and unit_name = 'KCAL'. Amounts are per 100 g. Refuse fraction lives in the food-portion/food tables.

The immediate deliverable is narrow: a caloric-yield-per-area table for the ~12 calorie-dense crops, computed from FAOSTAT yield × USDA energy
  density, to replace the blog-sourced table (patternbase) we cut.

  Concretely, one table with columns: crop | yield (kg/m²/yr, FAOSTAT) | energy density (kcal/kg, USDA) | caloric yield (kcal/m²/yr) | edible-adjusted. Covering tubers,
  cereals, pulses, and oilseeds/groundnuts.

  It grounds exactly three claims in the nutrition draft, all currently flagged % SOURCE NEEDED:
  1. The convergence — "the best calorie crops cluster around 1,500–2,000 kcal/m²/yr." The table shows the numbers and where they top out.
  2. The per-kg vs per-area distinction — oilseeds/groundnuts have high kcal/kg but not high kcal/m². This is why the table must include them: it's the evidence that
  energy-dense crops complement rather than shrink the area requirement.
  3. The area bound — 730,000 kcal/adult/yr ÷ best kcal/m²/yr → the ~200–470 m²/adult figure.

  That's the whole job for this dataset. The section's other quantitative anchors are already covered by sources we've verified: the photosynthetic ceiling (Zhu), the
  adult energy requirement (FAO/WHO/UNU), and the stacked-farming energy cost (Lovat/Milo). So the FAOSTAT/USDA work closes the last open % SOURCE NEEDED item — the
  area-bound numbers.

  One thing to keep in the back pocket, not the immediate goal: USDA carries the full nutrient profile, so the same pipeline can later compute micronutrient coverage per
  area for the "nutritional self-sufficiency" half of the section. But the table above is what we're actually after right now.

one more thing: where did we get the 12 calorie-dense crops? Since we have the usda data, couldn't we just look for the most calorie-dense crops on record?  

          

● Fair catch, and the honest answer is: we didn't. "~12 crops" was a placeholder I kept repeating; it was never derived from anything. The blog table had 6, I hand-waved
  12. So yes, let the data pick the crops instead of me guessing, that's more defensible. But there's one correction that matters, and it's the crux of the whole
  section.

  "Most calorie-dense" has two meanings, and they give different lists. If you rank the USDA data by kcal/100 g, the top of the list is cooking oils, nuts, and seeds
  (fat is 9 kcal/g). That's the per-kilogram ranking, and it's the wrong axis for the area-bound argument. The area bound is about kcal per square metre, which is yield 
  × energy density, and that ranking is dominated by high-yield crops: potato at 770 kcal/kg but 22 t/ha beats peanut at 5,670 kcal/kg but ~2 t/ha. So to find the most
  calorie-dense-per-area crops we can't look at USDA alone; we have to compute the join and rank by the result.

  Which actually makes your instinct better than my placeholder: compute kcal/m²/yr across all the crops we can match, and let the ranking name the crops. No a priori
  list.

  Two things to decide before I run it, because a blind top-N misleads:

  1. Produce both rankings, because their contrast is the evidence. Rank by kcal/kg (USDA) → oils, nuts, oilseeds. Rank by kcal/m²/yr (the join) → tubers, cereals. The
  gap between those two lists is the proof of the section's claim that energy-dense crops help storability, not area. So both rankings are outputs we want, not just the
  per-area one.
  2. A pure top-N by kcal/m² surfaces two edge cases at the very top: sugarcane and oil palm. They genuinely have the highest calories per hectare of anything, higher
  than cassava, but they're industrial crops (perennial, tropical), not dietary staples and not CEA candidates. So the ranking needs a scope: either exclude non-staple
  industrial crops, or include them and flag them explicitly as "highest per area but not food-staple / not CEA-relevant." I'd lean toward showing them with a flag,
  because "even the calorie-densest crops on record only reach X per m²" is a stronger honest bound than quietly dropping them.

  The one unavoidable manual step is still the crosswalk: FAOSTAT and USDA share no key, so matching an item to its energy density is by hand. But we only match the
  crops that plausibly rank high, not all 278, and the ranking tells us if we missed something surprising.

  So the plan I'd run once you've got pandas in: compute kcal/m²/yr for the candidate food crops from the join, rank them, produce the per-kg ranking alongside, flag
  sugarcane/oil-palm, and the crop list for the table falls out of the data instead of my guess. Want me to go with "include-and-flag" for the industrial crops, or
  restrict to edible staples from the start?

Schemas:
- faostat.*  -> FAOSTAT QCL: all_data_normalized (+ areacodes, elements, itemcodes, flags)
- usda_sr.*  -> USDA FoodData Central, SR Legacy (2018)
- usda_ff.*  -> USDA FoodData Central, Foundation Foods

In [7]:
import duckdb
import pandas as pd

In [ ]:
con = duckdb.connect("../food_data.duckdb", read_only=True)   

In [9]:
query1 = """
SELECT *
FROM faostat.all_data_normalized
WHERE Area = 'World' AND Element = 'Yield' AND Year = 2022
LIMIT 10;
"""


In [11]:
test_df = con.sql(query1).df()

In [12]:
test_df

,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
0,5000,'001,World,809,'01929.07,"Abaca, manila hemp, raw",5412,Yield,2022,2022,kg/ha,669.7,E,None
1,5000,'001,World,800,'01929.06,"Agave fibres, raw, n.e.c.",5412,Yield,2022,2022,kg/ha,680.4,E,None
2,5000,'001,World,221,'01371,"Almonds, in shell",5412,Yield,2022,2022,kg/ha,1563.8,A,None
3,5000,'001,World,711,'01654,"Anise, badian, coriander, cumin, caraway, fenn...",5412,Yield,2022,2022,kg/ha,1025.8,A,None
4,5000,'001,World,515,'01341,Apples,5412,Yield,2022,2022,kg/ha,20547.9,A,None
5,5000,'001,World,526,'01343,Apricots,5412,Yield,2022,2022,kg/ha,6693.3,E,None
6,5000,'001,World,226,'01379.01,Areca nuts,5412,Yield,2022,2022,kg/ha,2100.4,A,None
7,5000,'001,World,366,'01216,Artichokes,5412,Yield,2022,2022,kg/ha,13855.8,A,None
8,5000,'001,World,367,'01211,Asparagus,5412,Yield,2022,2022,kg/ha,5663.7,E,None
9,5000,'001,World,572,'01311,Avocados,5412,Yield,2022,2022,kg/ha,10483.1,A,None


In [ ]:
query_2 = 